In [5]:
import pandas as pd
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed,ProcessPoolExecutor
from urllib.parse import quote_plus
from rdflib.namespace import RDF, DC, Namespace
import xml.etree.ElementTree as ET
from lxml import etree
import shutil
import zipfile
import ftplib
import io
import csv
import field_extractor3 as fe

# Constants
UNZIP_DIR = "selected_data"
FTP_HOST = "download.europeana.eu"
FTP_PATH = "dataset/XML/"
OUTPUT_DIR = "collected_data"

In [6]:
# Open and read the CSV file
with open('/home/sbasir/Thesis/Thesis/EDP/sample_data/datasets.csv', newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)
    
    # Extract 'ids' and 'lang' and save them in a dictionary
    data_dict = {row['ids'] + '.zip': row['lang'] for row in reader}

    # Reset the reader to start extracting only 'ids' in a separate list
    csvfile.seek(0)  # Rewind the CSV file
    next(reader)  # Skip the header
    
    # Extract only 'ids' in a separate list
    data_ids = [row['ids'] + '.zip' for row in reader]

# Print the dictionary to verify
print(data_dict)

data_ids = data_ids[36]
print(data_ids)

{'00101.zip': 'pt', '00718.zip': 'de', '00719.zip': 'de', '00720.zip': 'de', '00721.zip': 'de', '00722.zip': 'de', '00723.zip': 'de', '00724.zip': 'de', '00725.zip': 'de', '00732.zip': 'de', '00733.zip': 'de', '00735.zip': 'de', '00736.zip': 'de', '00737.zip': 'de', '00738.zip': 'de', '00739.zip': 'de', '00740.zip': 'de', '00741.zip': 'de', '00742.zip': 'de', '00743.zip': 'de', '00744.zip': 'de', '00745.zip': 'de', '00746.zip': 'de', '00747.zip': 'de', '00902.zip': 'de', '02030.zip': 'pt', '02301.zip': 'it', '02302.zip': 'it', '03706.zip': 'fr', '03707.zip': 'fr', '03709.zip': 'fr', '03710.zip': 'fr', '03915.zip': 'fr', '03916.zip': 'fr', '03919.zip': 'fr', '03924.zip': 'fr', '03927.zip': 'fr', '03928.zip': 'fr', '03929.zip': 'fr', '03930.zip': 'fr', '04802.zip': 'fr', '05813.zip': 'ro', '05815.zip': 'ro', '05816.zip': 'ro', '07101.zip': 'sk', '07931.zip': 'de', '07932.zip': 'de', '08001.zip': 'fr', '08520.zip': 'sl', '08534.zip': 'pl', '08535.zip': 'de', '08547.zip': 'de', '08604.zip'

In [7]:
import os
import random
import lxml.etree as ET

def download_file(ftp_host, ftp_path, filename):
    zip_data = io.BytesIO()

    with ftplib.FTP(ftp_host) as ftp:
        ftp.login()  # Login as anonymous
        ftp.cwd(ftp_path)

        ftp.retrbinary(f'RETR {filename}', zip_data.write)
    
    zip_data.seek(0)
    return zip_data

# Function to unzip a file
def unzip_file(zip_data):
    extracted_files = []  # List to store file content

    with zipfile.ZipFile(zip_data, 'r') as zip_ref:
        for file_info in zip_ref.infolist():
            with zip_ref.open(file_info) as file:
                file_content = file.read()
                extracted_files.append(file_content)

    return extracted_files

def keep_random_10_percent(files):
    # Calculate 10% of the total number of files
    num_files_to_keep = max(1, int(len(files) * 0.15))  # Ensure at least one file is kept
    print(f"keep {num_files_to_keep} documents")

    # Randomly sample 10% of the files to keep
    files_to_keep = random.sample(files, num_files_to_keep)

    return files_to_keep

# Function to process a single subdirectory
def process_single_subdirectory(subdirectory_path):
    try:
        subdirectory = os.path.basename(subdirectory_path)
        print(f"Processing {subdirectory}...")

        # Get all files in the subdirectory
        files = os.listdir(subdirectory_path)
        size = len(files)
        print(f"Size of {subdirectory}: {size}")

        # Process RDF files in the sampled list
        output = fe.parse_rdf_files(subdirectory_path, subdirectory)
        
        # Write the output to an XML file
        output_file = f'/home/sbasir/Thesis/Thesis/EDP/collected_data/{subdirectory}'
        fe.write_data(output, output_file)

        print(f"Finished processing {subdirectory}. Output written to {output_file}")

        # Delete the subdirectory
        shutil.rmtree(subdirectory_path)
        print(f"Deleted subdirectory {subdirectory_path}")

    except Exception as e:
        print(f"Error processing {subdirectory}: {e}")

def process_zip_threaded(directory):
    subdirectories = [os.path.join(directory, subdir) for subdir in os.listdir(directory) if os.path.isdir(os.path.join(directory, subdir))]

    with ProcessPoolExecutor(max_workers=10) as executor:  # Experiment with the number of workers
        futures = {executor.submit(process_single_subdirectory, subdir): subdir for subdir in subdirectories}

        # Use tqdm to show progress as futures are completed
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing subdirectories"):
            try:
                future.result()
            except Exception as e:
                subdirectory = futures[future]
                print(f"Error processing subdirectory {subdirectory}: {e}")

In [8]:
from tqdm import tqdm  # Import tqdm for progress bar

def download_parsed_data(filename):
    try:
        print(f"Starting download and processing for {filename}...")

        # Construct the file path for the translation CSV
        translation_subdirectory = filename.replace(".zip", ".csv")
        csv_path = f'/home/sbasir/Thesis/Thesis/EDP/sample_data/translations/{translation_subdirectory}'
        
        # Check if the CSV file exists
        if not os.path.exists(csv_path):
            lang = data_dict.get(filename)
            if lang != "en":
                print(f"not considering {filename}")
                return
            
        # Download the ZIP file into memory
        zip_data = download_file(FTP_HOST, FTP_PATH, filename)

        # Unzip and process the file
        extracted_files = unzip_file(zip_data)
        print(f"first file: {extracted_files[0]}")
        print(f"deleting zip file {filename}")
        del zip_data 

        samples = keep_random_10_percent(extracted_files)
        del extracted_files

        total_samples = len(samples)  # Get total sample count for progress calculation

        # Initialize tqdm progress bar
        with tqdm(total=total_samples, desc=f"Processing {filename}") as pbar:
            # Process and write each document one by one
            for i, sample in enumerate(samples):
                parsed = fe.parse_file(sample)

                # Write each document to the output file one by one
                output_file = f'/home/sbasir/Thesis/Thesis/EDP/testing/{filename}/{i}.xml'
                fe.write_data([parsed], output_file)  # Write one parsed document at a time

                # Update tqdm progress bar
                pbar.update(1)

        del samples

        print(f"Finished processing {filename}")

    except Exception as e:
        print(f"Error processing {filename}: {e}")


In [9]:
download_parsed_data(data_ids)

Starting download and processing for 03927.zip...
first file: b'<?xml version="1.0" encoding="UTF-8" standalone="yes"?><rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:dcterms="http://purl.org/dc/terms/" xmlns:edm="http://www.europeana.eu/schemas/edm/" xmlns:owl="http://www.w3.org/2002/07/owl#" xmlns:wgs84_pos="http://www.w3.org/2003/01/geo/wgs84_pos#" xmlns:skos="http://www.w3.org/2004/02/skos/core#" xmlns:rdaGr2="http://rdvocab.info/ElementsGr2/" xmlns:foaf="http://xmlns.com/foaf/0.1/" xmlns:ebucore="http://www.ebu.ch/metadata/ontologies/ebucore/ebucore#" xmlns:doap="http://usefulinc.com/ns/doap#" xmlns:odrl="http://www.w3.org/ns/odrl/2/" xmlns:cc="http://creativecommons.org/ns#" xmlns:ore="http://www.openarchives.org/ore/terms/" xmlns:svcs="http://rdfs.org/sioc/services#" xmlns:oa="http://www.w3.org/ns/oa#" xmlns:dqv="http://www.w3.org/ns/dqv#"><edm:ProvidedCHO rdf:about="http://data.europeana.eu/item/03927/_0012_oai_

Processing 03927.zip:  58%|█████▊    | 1930/3325 [00:48<00:35, 38.94it/s]